# Equipo 19 · Avance 4: Endo-STTN y el estimador de profundidad entrenados juntos (modelo E)

En el Avance 2, Endo-STTN está **congelado** (modelo **D**). Aquí Endo-STTN y AF-SfMLearner forman un solo modelo y se
entrenan **juntos**: la pérdida de profundidad también ajusta el relleno de los reflejos.

| Paso | Script | Qué hace |
|---|---|---|
| 1 | `codigo/preparar_datos.py` | Los **mismos videos** y la misma división del Avance 2 (se reutilizan desde Drive) |
| 2 | `codigo/limpiar_endosttn.py` | `rgb_limpio` con Endo-STTN original: lo necesita D y es el ancla/referencia de E |
| 3 | `codigo/ajustar_profundidad.py` | Modelos C y D (se reutilizan desde Drive si ya existen) |
| 4 | `Avance4_EntrenamientoConjunto/ajustar_conjunto.py` | **Modelo E**: Endo-STTN + AF-SfMLearner de punta a punta |
| 5 | `codigo/evaluar_profundidad.py` | Compara A–E contra la profundidad real; la prueba clave es **D vs E** |

**Hipótesis:** E tiene menor error de profundidad que D, sobre todo dentro de los reflejos, sin que el relleno deje de
parecerse a tejido. Detalle en [`Avance4_EntrenamientoConjunto/README.md`](../README.md).

## 0. Preparar Colab

Activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU*; si tienes acceso, una L4 o A100
va mejor porque E necesita más memoria). En `MyDrive/Equipo19/` se reutiliza lo que dejó el notebook del Avance 2:

| En Drive | Para qué | Si no está |
|---|---|---|
| `c3vd_preparado.zip` o `raw/` | Mismos videos de C3VDv2 | Hace falta uno de los dos |
| `c3vd_limpio.zip` | Frames limpios del Avance 2 (lo crea este notebook) | Se corre la limpieza (paso 2) |
| `modelos/rgb_s0`, `modelos/rgb_limpio_s0` | Modelos C y D del Avance 2 | Se entrenan aquí (paso 3) |
| `gen_00009.pth`, `Model_trained_end_to_end.zip` | Pesos | Se descargan de la carpeta del equipo |

Al final se guardan en Drive `modelos/` (incluido `conjunto_s0/`) y `resultados_avance4/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, shutil, subprocess

# ---------- Ajusta si hace falta ----------
EPOCAS, SEMILLA = 15, 0
REENTRENAR_CD = False        # True: vuelve a entrenar C y D aunque estén en Drive
CORRER_E      = True
CORRER_EVALUAR = True
# Modelo E (ver Avance4_EntrenamientoConjunto/README.md)
STTN_CAPAS = "todo"          # "todo" o "transformer" (menos memoria, cambia menos el relleno)
PESO_HUECO = 1.0             # reconstrucción de tejido sano con la máscara desplazada
PESO_ANCLA = 0.0             # parecido al relleno del Endo-STTN original (p. ej. 0.5)
LOTE_E     = 2               # clips de 5 frames por lote; baja a 1 si falta memoria
DIVISION_MANUAL = None       # igual que en el Avance 2; None = automática
DRIVE = Path("/content/drive/MyDrive/Equipo19")
REPO_URL = "https://github.com/Jkalks2025/Eliminaci-n-de-artefactos-en-im-genes-de-endoscop-a.git"
# ------------------------------------------

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")

REPO = Path("/content/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate"], check=True)
(REPO / "externos").mkdir(exist_ok=True)
if (DRIVE / "Model_trained_end_to_end.zip").exists():
    shutil.copy(DRIVE / "Model_trained_end_to_end.zip", REPO / "externos")
pre = REPO / "Endo-STTN" / "release_model" / "pretrained_model"
if (DRIVE / "gen_00009.pth").exists() and not (pre / "gen_00009.pth").exists():
    pre.mkdir(parents=True, exist_ok=True); shutil.copy(DRIVE / "gen_00009.pth", pre)

# Datos: los mismos del Avance 2 (preparados en zip o ZIPs crudos de C3VDv2)
datos = REPO / "datos"
CORRER_PREPARAR = True
if (DRIVE / "c3vd_preparado.zip").exists() and not (datos / "c3vd" / "division.json").exists():
    shutil.unpack_archive(str(DRIVE / "c3vd_preparado.zip"), str(datos))
if (datos / "c3vd" / "division.json").exists():
    CORRER_PREPARAR = False
elif (DRIVE / "raw").exists():
    (datos / "raw").mkdir(parents=True, exist_ok=True)
    for z in (DRIVE / "raw").glob("*.zip"):
        destino = datos / "raw" / z.name
        if not destino.exists():
            os.symlink(z, destino)            # se leen directo de Drive, sin copiar los GB
if (DRIVE / "c3vd_limpio.zip").exists():       # frames limpios guardados por este notebook
    shutil.unpack_archive(str(DRIVE / "c3vd_limpio.zip"), str(datos))
if (DRIVE / "modelos").exists():               # C y D del Avance 2
    shutil.copytree(DRIVE / "modelos", REPO / "modelos", dirs_exist_ok=True)
subprocess.run([sys.executable, str(REPO / "preparar_entorno.py")], check=True)

In [ ]:
os.chdir(REPO)
# Los resultados de este avance van a su propia carpeta (no se mezclan con los del Avance 2)
os.environ["EQUIPO19_RESULTADOS"] = str(REPO / "Avance4_EntrenamientoConjunto" / "resultados")
sys.path.insert(0, str(REPO / "codigo"))
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import c3vd, modelo_profundidad as mp

def correr(*args):
    """Corre un script y muestra su salida línea por línea mientras avanza."""
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=dict(os.environ, PYTHONWARNINGS="ignore"))
    for linea in p.stdout:
        print(linea, end="", flush=True)
    assert p.wait() == 0, f"falló: {args}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Repositorio :", REPO)
if DEVICE == "cuda":
    print("Dispositivo :", torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)")
else:
    print("Dispositivo : cpu (sin GPU: entrenar E será muy lento)")
print("Modelos del Avance 2 encontrados:", sorted(p.name for p in c3vd.DIR_MODELOS.glob("*_s*")) if c3vd.DIR_MODELOS.exists() else "ninguno")

## 1. Datos: los mismos videos del Avance 2

Si `c3vd_preparado.zip` estaba en Drive, este paso se salta. La división por video es la misma, así que C, D y E se
entrenan, validan y prueban con los mismos frames.

In [ ]:
if CORRER_PREPARAR:
    correr("codigo/probar_formato.py")
    correr("codigo/preparar_datos.py")
    shutil.make_archive(str(DRIVE / "c3vd_preparado"), "zip", root_dir=REPO / "datos", base_dir="c3vd")

assert (c3vd.DIR_DATOS / "division.json").exists(), "Faltan los datos: pon c3vd_preparado.zip o raw/ en Drive"
if DIVISION_MANUAL:
    json.dump(DIVISION_MANUAL, open(c3vd.DIR_DATOS / "division.json", "w"), indent=2)
division = json.load(open(c3vd.DIR_DATOS / "division.json"))
for k, v in division.items():
    print(f"{k:14s} {len(v):2d} videos: {', '.join(v)}")

## 2. Frames limpios con el Endo-STTN original

Los usa el modelo D y, en E, sirven de referencia: el **cambio del relleno** mide cuánto se aleja el Endo-STTN
ajustado del original. Solo se limpian los videos que falten; el resultado se guarda en Drive (`c3vd_limpio.zip`).

In [ ]:
faltan = [v for v in c3vd.videos_preparados() if not (c3vd.DIR_DATOS / v / "rgb_limpio.npy").exists()]
if faltan:
    print("Limpiando", len(faltan), "videos")
    correr("codigo/limpiar_endosttn.py")
    temp = Path("/content/c3vd_limpio")
    for v in c3vd.videos_preparados():
        (temp / "c3vd" / v).mkdir(parents=True, exist_ok=True)
        shutil.copy(c3vd.DIR_DATOS / v / "rgb_limpio.npy", temp / "c3vd" / v)
    shutil.make_archive(str(DRIVE / "c3vd_limpio"), "zip", root_dir=temp, base_dir="c3vd")
    print("Guardado en", DRIVE / "c3vd_limpio.zip")
else:
    print("Todos los videos ya tienen rgb_limpio.npy")

## 3. Modelos C y D (Avance 2)

Si ya están en Drive (`modelos/rgb_s<semilla>` y `modelos/rgb_limpio_s<semilla>`) se reutilizan; si no, se entrenan
con la misma configuración del Avance 2.

In [ ]:
for entrada in ["rgb", "rgb_limpio"]:
    carpeta = c3vd.DIR_MODELOS / f"{entrada}_s{SEMILLA}"
    if REENTRENAR_CD or not (carpeta / "encoder.pth").exists():
        correr("codigo/ajustar_profundidad.py", "--entrada", entrada, "--epocas", EPOCAS, "--semilla", SEMILLA)
    else:
        print(f"{carpeta.name}: se reutiliza el modelo de Drive")

## 4. Modelo E: Endo-STTN + estimador de punta a punta

Pérdida: $\mathcal{L}_{SI} + \lambda_h \mathcal{L}_{hueco} + \lambda_a \mathcal{L}_{ancla}$. En cada época se reporta el
AbsRel de validación y el **cambio del relleno** (0–255) respecto al Endo-STTN original. Si sale un error de memoria
(`CUDA out of memory`), baja `LOTE_E` a 1 o usa `STTN_CAPAS = "transformer"`.

In [ ]:
if CORRER_E:
    correr("Avance4_EntrenamientoConjunto/ajustar_conjunto.py", "--epocas", EPOCAS, "--semilla", SEMILLA,
           "--lote", LOTE_E, "--sttn-capas", STTN_CAPAS, "--peso-hueco", PESO_HUECO, "--peso-ancla", PESO_ANCLA)
    shutil.copytree(c3vd.DIR_MODELOS / f"conjunto_s{SEMILLA}", DRIVE / "modelos" / f"conjunto_s{SEMILLA}", dirs_exist_ok=True)

In [ ]:
hist = {nombre: json.load(open(c3vd.DIR_MODELOS / carpeta / "historial.json"))["epocas"]
        for nombre, carpeta in [("C · originales", f"rgb_s{SEMILLA}"), ("D · limpios", f"rgb_limpio_s{SEMILLA}"),
                                ("E · conjunto", f"conjunto_s{SEMILLA}")]
        if (c3vd.DIR_MODELOS / carpeta / "historial.json").exists()}
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
for nombre, ep in hist.items():
    ax[0].plot([e["epoca"] for e in ep], [e["abs_rel_val"] for e in ep], "o-", label=nombre)
ax[0].set_title("AbsRel de validación"); ax[0].set_xlabel("época"); ax[0].legend(); ax[0].grid(alpha=.3)
if "E · conjunto" in hist:
    ep = hist["E · conjunto"]
    ax[1].plot([e["epoca"] for e in ep], [e["cambio_relleno_val"] for e in ep], "o-", color="C2")
    ax[1].set_title("E: cambio del relleno vs. Endo-STTN original (0-255)"); ax[1].set_xlabel("época"); ax[1].grid(alpha=.3)
    for k in ["prof", "hueco", "ancla"]:
        ax[2].plot([e["epoca"] for e in ep[1:]], [e[f"perdida_{k}"] for e in ep[1:]], "o-", label=k)
    ax[2].set_title("E: pérdidas de entrenamiento"); ax[2].set_xlabel("época"); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## 5. Evaluación en videos no vistos

| Modelo | Pesos | Frames de prueba |
|---|---|---|
| A | AF-SfMLearner (SCARED) | originales |
| B | AF-SfMLearner (SCARED) | limpios |
| C | ajustado con originales | originales |
| D | ajustado con limpios | limpios (Endo-STTN original) |
| **E** | **Endo-STTN + estimador ajustados juntos** | **limpios por su Endo-STTN ajustado** |

La comparación decisiva es **D vs E** (Wilcoxon pareado por frame): solo cambia que Endo-STTN también se entrenó.

In [ ]:
if CORRER_EVALUAR:
    correr("codigo/evaluar_profundidad.py", "--semilla", SEMILLA)
from IPython.display import Markdown, display
display(Markdown(open(c3vd.DIR_RESULTADOS / "evaluacion_resumen.md").read()))

### ¿El relleno de E sigue pareciendo tejido?

Frame de prueba con más reflejos: original, relleno del Endo-STTN original (D), relleno del Endo-STTN ajustado (E) y la
diferencia entre ambos. Si E gana en profundidad pero su relleno se ve artificial, la mejora viene de que el relleno
"codifica" profundidad y no de un mejor inpainting.

In [ ]:
video = division["prueba"][0]
rgb, prof, mask = c3vd.cargar_video(video)
i = int(np.argmax(np.asarray(mask).reshape(len(mask), -1).mean(1)))   # frame con más reflejos
limpio_d = c3vd.cargar_video(video, "rgb_limpio")[0][i]
ruta_e = c3vd.DIR_DATOS / video / f"rgb_conjunto_s{SEMILLA}.npy"
limpio_e = np.load(ruta_e, mmap_mode="r")[i] if ruta_e.exists() else None

ys, xs = np.nonzero(mask[i]) if mask[i].any() else (np.array([128]), np.array([160]))
cy, cx = int(np.median(ys)), int(np.median(xs))
y0, x0 = np.clip(cy - 48, 0, rgb.shape[1] - 96), np.clip(cx - 60, 0, rgb.shape[2] - 120)
recorte = lambda a: a[y0:y0 + 96, x0:x0 + 120]
imgs = [(rgb[i], "original"), (limpio_d, "D: Endo-STTN original")]
if limpio_e is not None:
    dif = np.abs(limpio_e.astype(int) - limpio_d.astype(int)).mean(-1)
    imgs += [(limpio_e, "E: Endo-STTN ajustado"), (dif, f"|E − D| (media en reflejos {dif[mask[i]].mean():.1f})")]
fig, ax = plt.subplots(2, len(imgs), figsize=(4.5 * len(imgs), 7))
for j, (img, t) in enumerate(imgs):
    kw = dict(cmap="inferno", vmin=0, vmax=60) if img.ndim == 2 else {}
    ax[0, j].imshow(img, **kw); ax[0, j].set_title(f"{t}\n{video} · frame {i}", fontsize=9)
    ax[1, j].imshow(recorte(img), **kw); ax[1, j].set_title("zoom en el reflejo", fontsize=9)
ax[0, 0].add_patch(plt.Rectangle((x0, y0), 120, 96, fill=False, color="cyan"))
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
modelos = {"A": (mp.PESOS_SCARED, "rgb"), "B": (mp.PESOS_SCARED, "rgb_limpio"),
           "C": (c3vd.DIR_MODELOS / f"rgb_s{SEMILLA}", "rgb"), "D": (c3vd.DIR_MODELOS / f"rgb_limpio_s{SEMILLA}", "rgb_limpio"),
           "E": (c3vd.DIR_MODELOS / f"conjunto_s{SEMILLA}", f"rgb_conjunto_s{SEMILLA}")}
g = np.asarray(prof[i], np.float32); valido = g > 0
fig, ax = plt.subplots(2, 6, figsize=(24, 7))
ax[0, 0].imshow(rgb[i]); ax[0, 0].set_title("frame original")
ax[1, 0].imshow(np.where(valido, g, np.nan), cmap="magma_r"); ax[1, 0].set_title("profundidad real")
for j, (nombre, (pesos, entrada)) in enumerate(modelos.items(), 1):
    if not (Path(pesos) / "encoder.pth").exists() or not (c3vd.DIR_DATOS / video / f"{entrada}.npy").exists():
        ax[0, j].set_title(f"{nombre}: sin modelo"); continue
    enc, dec = mp.cargar(pesos, DEVICE); enc.eval(); dec.eval()
    x = c3vd.cargar_video(video, entrada)[0][i:i + 1]
    with torch.no_grad():
        p = mp.predecir(enc, dec, mp.a_tensor(x, DEVICE))[0, 0].cpu().numpy()
    p = p * np.median(g[valido]) / np.median(p[valido])
    err = np.where(valido, np.abs(p - g) / np.maximum(g, 1e-3), np.nan)
    ax[0, j].imshow(p, cmap="magma_r"); ax[0, j].set_title(f"{nombre}: predicción")
    ax[1, j].imshow(err, cmap="inferno", vmin=0, vmax=0.5); ax[1, j].set_title(f"{nombre}: error relativo")
    ax[1, j].contour(mask[i], [0.5], colors="cyan", linewidths=0.6)
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Guardar modelos y resultados en Drive
shutil.copytree(c3vd.DIR_MODELOS, DRIVE / "modelos", dirs_exist_ok=True)
if c3vd.DIR_RESULTADOS.exists():
    shutil.copytree(c3vd.DIR_RESULTADOS, DRIVE / "resultados_avance4", dirs_exist_ok=True)
print("Copiado a", DRIVE)

## 6. Cómo leer los resultados

- **E < D en AbsRel dentro de los reflejos** (Wilcoxon p < 0.05), sin empeorar en toda la imagen y con un relleno que
  sigue pareciendo tejido: entrenar Endo-STTN junto con el estimador ayuda. Es el resultado buscado.
- **E < D, pero el relleno de E se ve artificial** (cambio del relleno alto, |E − D| grande): Endo-STTN aprendió a
  "pintar pistas" para el estimador. Repite con `PESO_HUECO = 2` o `PESO_ANCLA = 0.5` y compara.
- **E ≈ D**: el Endo-STTN original ya da un relleno suficiente; prueba `STTN_CAPAS = "transformer"` o más épocas.
- **E > D**: el ajuste conjunto desestabiliza el entrenamiento; baja el lr de Endo-STTN (`--lr-sttn 1e-6`).

Repite con otras semillas (`SEMILLA = 1, 2`): C y D también deben existir para esa semilla.